graph TD
    %% Định nghĩa style cho các khối
    classDef default fill:#f9f9f9,stroke:#333,stroke-width:2px;
    classDef agent fill:#e0f2fe,stroke:#0284c7,stroke-width:2px;
    classDef tool fill:#fefce8,stroke:#ca8a04,stroke-width:2px;
    classDef report fill:#ecfdf5,stroke:#16a34a,stroke-width:2px;
    classDef io fill:#f3e8ff,stroke:#8b5cf6,stroke-width:2px;

    %% STAGE 1: Giao diện & Nhận yêu cầu
    subgraph 1. Giao diện Người dùng & Nhận Yêu cầu
        User("👨‍💻 Nhà đầu tư") --> UI["🌐 Giao diện Web (Flask/SocketIO)"];
        UI --> Query["📝 Nhập yêu cầu<br/>'Phân tích HPG và file BCTC quý 2'"];
        UI --> PDF_Input["📄 Tải lên file BCTC.pdf"];
    end

    %% STAGE 2: Phân tích Ý định & Điều phối
    subgraph 2. API Backend & Phân tích Ý định
        Query & PDF_Input --> API["🚀 Flask Server (api.py)"];
        API -- "1. Gửi yêu cầu" --> IntentRecognition["🧠 <b>Mistral AI</b><br/>(get_user_intent_with_mistral)"];
        IntentRecognition -- "2. Trả về Intent có cấu trúc" --> Intent["📜 <b>Intent Object</b><br/>{task: 'comprehensive_analysis',<br/>ticker: 'HPG',<br/>file_path: 'uploads/BCTC.pdf'}"];
    end

    %% STAGE 3: Lõi Phân tích Multi-Agent
    subgraph 3. Lõi Phân tích Multi-Agent (CrewAI)
        Intent -- "3. Khởi chạy Crew" --> Crew["🤖 <b>Financial Crew</b> (main.py)"];

        %% Quản lý Key
        KeyManager["🔑 <b>Key Manager</b><br/>(key_manager.py)<br/>Xoay vòng API Keys"]

        %% Nhánh Agent 1: Vĩ mô
        subgraph "Agent 1: Chuyên gia Tin tức"
            Agent1("📰 Market News Analyst"):::agent;
            Tools1("🛠️ Search & Scrape Tools"):::tool;
            Source1["🌍 Google Search"];
            Report1("📄 Báo cáo Vĩ mô & Thị trường"):::report;
        end
        Crew --> Agent1;
        KeyManager -.-> Agent1;
        Agent1 --> Tools1 --> Source1;
        Agent1 --> Report1;

        %% Nhánh Agent 2: Kỹ thuật
        subgraph "Agent 2: Chuyên gia Kỹ thuật"
            Agent2("📈 Technical Analyst"):::agent;
            Tools2("🛠️ TechDataTool"):::tool;
            Source2["📦 vnstock (Dữ liệu giá)"];
            Report2("📄 Báo cáo Phân tích Kỹ thuật HPG"):::report;
        end
        Crew --> Agent2;
        KeyManager -.-> Agent2;
        Agent2 --> Tools2 --> Source2;
        Agent2 --> Report2;

        %% Nhánh Agent 3: Cơ bản & Đối thủ
        subgraph "Agent 3: Chuyên gia Cơ bản"
            Agent3("🏢 Financial & Competitor Analyst"):::agent;
            Tools3("🛠️ ComprehensiveFinancialTool"):::tool;
            Source3a["📦 vnstock (Dữ liệu TC)"];
            Source3b["🧠 Knowledge Base<br/>(PE_PB_industry_average.json)"];
            Report3("📄 Báo cáo Cơ bản & So sánh Đối thủ HPG"):::report;
        end
        Crew --> Agent3;
        KeyManager -.-> Agent3;
        Agent3 --> Tools3;
        Tools3 --> Source3a;
        Tools3 --> Source3b;
        Agent3 --> Report3;

        %% Nhánh Agent 4: PDF
        subgraph "Agent 4: Chuyên gia BCTC"
            Agent4("📑 PDF Report Analyst"):::agent;
            Tools4("🛠️ MistralOCRTool, FileReadTool"):::tool;
            Source4["📄 File BCTC.pdf từ người dùng"];
            Report4("📄 Tóm tắt Báo cáo Tài chính HPG"):::report;
        end
        Crew --> Agent4;
        KeyManager -.-> Agent4;
        Agent4 --> Tools4 --> Source4;
        Agent4 --> Report4;
    end

    %% STAGE 4: Tổng hợp & Tạo Báo cáo cuối cùng
    subgraph 4. Tổng hợp & Tạo Báo cáo
        AgentEditor("👑 <b>Report Editor Agent</b> (Tổng biên tập)"):::agent;
        KeyManager -.-> AgentEditor;
        Report1 & Report2 & Report3 & Report4 -- "4. Tổng hợp tất cả thông tin" --> AgentEditor;
        AgentEditor -- "5. Viết báo cáo hoàn chỉnh" --> FinalReport["⭐ <b>Bản tin Chứng khoán HPG.md</b><br/>(Báo cáo cuối cùng)"]:::report;
    end
    
    %% STAGE 5: Trả kết quả
    subgraph 5. Trả kết quả về Giao diện
        FinalReport -- "6. Lưu file & gửi qua Socket.IO" --> API;
        API -- "7. Hiển thị kết quả" --> UI;
    end

    %% Class Styles
    class User,UI,Query,PDF_Input,API,IntentRecognition,Intent io;